In [1]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../Dataset/cleaned-dataset/merged-dataset-cleaned.parquet")

### 1. Log Flux

In [2]:
df["log_flux"] = np.log10(df["E2W_COR_FLUX"] + 1) # +1 to prevent divide by 0 error

### Target Variable: Log Flux 1-Hour Shifted

In [3]:
df["target_log_flux_1h"] = df["log_flux"].shift(-60)

### lag Features: t-15m, t-30m, t-60m
log_flux, F, BX_GSE, BY_GSM, BZ_GSM, flow_speed, proton_density

In [4]:
all_features = ["log_flux", "F", "BX_GSE", "BY_GSM", "BZ_GSM", "flow_speed", "proton_density"]
lags = [15, 30, 60]

for col in all_features:
    for lag in lags:
        df[f"{col}_lag_{lag}m"] = df[col].shift(lag)

### Rolling Window Features (1h and 3h window)
BZ_GSM, flow_speed, log_flux

In [5]:
rolling_targets = ["BZ_GSM", "flow_speed", "log_flux"]

for col in rolling_targets:
    # 1-Hour Rolling Average & Standard Deviation
    df[f"{col}_roll_mean_1h"] = df[col].rolling(window=60, min_periods=30).mean()
    df[f"{col}_roll_std_1h"]  = df[col].rolling(window=60, min_periods=30).std()
    
    # 3-Hour Rolling Average
    df[f"{col}_roll_mean_3h"] = df[col].rolling(window=180, min_periods=90).mean()

### Saving the final dataset

In [6]:
df.to_parquet("../Dataset/final-dataset/dataset-with-features.parquet", engine="pyarrow")
print("Successfully saved the final dataset with egineered features.")

Successfully saved the final dataset with egineered features.
